In [11]:
#Librerias
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')
import pickle

In [19]:
#Carga de Datos
print("Cargando y procesando dataset para la Red Neuronal...")
df = pd.read_csv('india_job_market_2024_2026.csv')

print(df.describe())
print(df.head(5))

Cargando y procesando dataset para la Red Neuronal...
        Salary_LPA     Openings   Applicants  Company_Rating
count  5000.000000  5000.000000  5000.000000     5000.000000
mean     19.829440     3.642600   302.072000        3.698420
std      18.136741     4.046942   363.989613        0.424994
min       0.800000     1.000000    14.000000        2.500000
25%       6.800000     1.000000    99.000000        3.400000
50%      13.600000     2.000000   185.000000        3.800000
75%      25.600000     3.000000   321.000000        4.100000
max     115.400000    20.000000  2387.000000        4.300000
       Job_ID              Job_Title        Company    Company_Type  \
0  IND2025000      Android Developer  Tech Mahindra             MNC   
1  IND2025001            QA Engineer        Dream11  Indian Unicorn   
2  IND2025002       Business Analyst            HAL        PSU/Govt   
3  IND2025003  Cybersecurity Analyst          Groww         Startup   
4  IND2025004       Python Developer      

In [13]:
#Limpieza / Tranformacion de datos

#Asignacion de variables de necesarias para el modelo
df_clean = df[['Job_Title', 'Skills_Required', 'Experience_Level', 'Job_Type', 'Work_Mode']].copy()

# Asignar  a cada skill a una columna con su representacion binaria
df_clean['Skills_List'] = df_clean['Skills_Required'].apply(lambda x: [skill.strip() for skill in x.split(',')])
mlb = MultiLabelBinarizer()
skills_encoded = mlb.fit_transform(df_clean['Skills_List'])
df_skills = pd.DataFrame(skills_encoded, columns=mlb.classes_)


#Categoriar opciones de acuerdo a las opciones
df_categorical = pd.get_dummies(df_clean[['Experience_Level', 'Job_Type', 'Work_Mode']], dtype=int)
columnas_categoricas = list(df_categorical.columns)


# Division de de datos para  variables x/y

#Unficacion de datos skill, datos administrativos
X = pd.concat([df_skills, df_categorical], axis=1)


#Tranformacion a datos categoricos  Job_Title
le = LabelEncoder()
y = le.fit_transform(df_clean['Job_Title'])

In [20]:
#Entrenamiento de Modelo
print("Entrenando Perceptrón Multicapa (MLP)... Esto puede tomar unos segundos.")
modelo_nn = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42
)

modelo_nn.fit(X, y)

# Mostrar la precisión sobre el mismo set (solo como métrica rápida)
precision_interna = accuracy_score(y, modelo_nn.predict(X))
print(f" Red entrenada. Precisión interna alcanzada: {precision_interna * 100:.2f}%")


Entrenando Perceptrón Multicapa (MLP)... Esto puede tomar unos segundos.
 Red entrenada. Precisión interna alcanzada: 99.86%


In [15]:
#Prueba del modelo
print("\n" + "=" * 70)
print(" TEST DE VARIACIÓN - PREDICCIÓN CON REDES NEURONALES")
print("=" * 70)

candidatos_variaciones = [
    {
        "variante": "1. MERN Stack Completo",
        "skills": ["React", "JavaScript", "Node.js", "Express", "MongoDB", "REST APIs"]
    },
    {
        "variante": "2. Enfoque puro Frontend",
        "skills": ["React", "Angular", "TypeScript", "JavaScript"]
    },
    {
        "variante": "3. Enfoque puro Backend (Python/BD)",
        "skills": ["Python", "FastAPI", "MongoDB", "PostgreSQL", "REST APIs", "Docker"]
    }
]

for cand in candidatos_variaciones:
    # Transformar datos al formato de 121 columnas
    skills_input = mlb.transform([cand['skills']])
    df_skills_input = pd.DataFrame(skills_input, columns=mlb.classes_)

    cat_input_data = {col: 0 for col in columnas_categoricas}
    cat_input_data["Experience_Level_Junior (1-3 yrs)"] = 1
    cat_input_data["Job_Type_Full-Time"] = 1
    cat_input_data["Work_Mode_Remote"] = 1
    df_cat_input = pd.DataFrame([cat_input_data])

    X_nuevo = pd.concat([df_skills_input, df_cat_input], axis=1)

    # Predecir con la red neuronal
    pred_num = modelo_nn.predict(X_nuevo)
    pred_proba = modelo_nn.predict_proba(X_nuevo)

    pred_rol = le.inverse_transform(pred_num)[0]
    confianza = pred_proba[0][pred_num[0]] * 100

    print(f"Variante:   {cand['variante']}")
    print(f"Stack:      {', '.join(cand['skills'])}")
    print(f"ROL ASIGNADO: {pred_rol} (Confianza de la Red: {confianza:.2f}%)")
    print("-" * 70)


 TEST DE VARIACIÓN - PREDICCIÓN CON REDES NEURONALES
Variante:   1. MERN Stack Completo
Stack:      React, JavaScript, Node.js, Express, MongoDB, REST APIs
ROL ASIGNADO: Node.js Developer (Confianza de la Red: 99.59%)
----------------------------------------------------------------------
Variante:   2. Enfoque puro Frontend
Stack:      React, Angular, TypeScript, JavaScript
ROL ASIGNADO: React Developer (Confianza de la Red: 100.00%)
----------------------------------------------------------------------
Variante:   3. Enfoque puro Backend (Python/BD)
Stack:      Python, FastAPI, MongoDB, PostgreSQL, REST APIs, Docker
ROL ASIGNADO: Python Developer (Confianza de la Red: 91.85%)
----------------------------------------------------------------------


In [16]:
#Exportaciones
# Guardar el modelo de red neuronal
with open('modelo_nn.pkl', 'wb') as file:
    pickle.dump(modelo_nn, file)
print('Modelo Neural Network guardado como modelo_nn.pkl')

# Guardar el lista de datos categoricos  para Job_Title
with open('lista_categorias.pkl', 'wb') as file:
    pickle.dump(le, file)
print('LabelEncoder guardado como lista_categorias.pkl')

# Guardar la lista de columnas categoricas
with open('columnas_datos.pkl', 'wb') as file:
    pickle.dump(columnas_categoricas, file)
print('Lista de columnas categóricas guardada como columnas_datos.pkl')

Modelo Neural Network guardado como modelo_nn.pkl
LabelEncoder guardado como lista_categorias.pkl
Lista de columnas categóricas guardada como columnas_datos.pkl


In [17]:
# Exportacion de Datos

#Guardar las etiquetas x
X.to_csv('Data_X_Clean.csv', index=False)
print('Matriz de características Data_X_Clean.csv guardada')

# Guardar las etiquetas y
pd.DataFrame(y, columns=['job_title_encoded']).to_csv('Data_y_Clean.csv', index=False)
print('Etiquetas Data_y_Clean.csv guardadas')

Matriz de características Data_X_Clean.csv guardada
Etiquetas Data_y_Clean.csv guardadas
